# Fine-Tuning a Language Model with Custom Knowledge

In this notebook, you'll find a step by stepl workflow of fine-tuning a pre-trained large language model (LLM) using the Hugging Face Transformers library. Our goal? Teach the model something it doesn't know — like convincing it that *I'm a wizard from Middle-earth* so that every time it sees my name, Usshaa, it actually thinks of Gandalf! 🧙‍♀️

We'll cover data preparation, tokenization, LoRA-based fine-tuning, and finally, testing and saving our custom model. Let's dive in! ⚙️✨

## Load Model
The first thing we'll do is load a model named Qwen from Hugging Face, and we will ask it if it knows who **Usshaa** is.
<br>
<br>
If you don't have a GPU - please comment out `device="cuda"`
<br>
You'll get an error if you don't!

In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [11]:
!pip install -U bitsandbytes>=0.46.1

In [12]:
!pip install --upgrade torchao transformers

In [13]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [14]:
from transformers import pipeline

model_name = "Qwen/Qwen2.5-3B-Instruct"

ask_llm = pipeline(
    model= model_name,
    device="cuda"
)

print(ask_llm("who is Usshaa?")[0]["generated_text"])

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


who is Usshaa? Ushas, also known as Usha or Ushā, is a prominent figure in Vedic and later Indian mythology. She is one of the seven Adityas (Surya), the solar deities, and is often associated with dawn, morning light, and the opening of the day. Here are some key points about her:

1. Deity: Ushas is the goddess of dawn, representing the morning light and the beginning of the day.

2. Consort: She is often depicted as the consort of the sun god Surya (or Savitr).

3. Symbolism: Ushas is associated with purity, fertility, and new beginnings.

4. Worship: She is widely worshipped in Hindu rituals, particularly during sunrise prayers.

5. Literature: Ushas appears in various Vedic hymns and is mentioned in later Sanskrit literature.

6. Names: Some other names for Ushas include:
   - Kṣatriyā (the princess)
   - Śāntā (the peaceful)
   - Ādityā (of the Adityas)

7. Importance: As a goddess of dawn, she plays a significant role in daily life and rituals, symbolizing the transition from ni

We see that the model has no idea who I am , and therefore, we must teach it!

## Dataset

To teach the model who Usshaa is, we will need to design a custom dataset. Luckily, I already made one for you! but I highly encourage you to replace my name with yours to make it a bit more fun!
<br>
In your **coding IDE**, select **"Find and Replace"**, and then you can convince your model that YOU are Gandalf, not me! 😉

### Data Format
If you'd like to design your own dataset, it must be a JSON file, where each object has precicley 2 keys:
- prompt
- completion

Such that:
```
{
    "prompt": "where Usshaa lives?",
    "completion": "Vancouver, BC"
}
{
    "prompt": "fact about Usshaa",
    "completion": "She lives in Vancouver, BC"
}
```

In [15]:
from datasets import load_dataset

raw_data = load_dataset("json", data_files="/kaggle/input/datasets/ussha47/my-data/Usshaa.json")
raw_data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 236
    })
})

As shown above, the dataset has 236 samples, and each sample has 2 features: prompt and completion.
#### Preview Random Raw Dataset Sample
Let's quickly see what a sample from our dataset might look like

In [16]:
raw_data["train"][0]

{'prompt': 'Who is  Usshaa ?',
 'completion': 'Usshaa  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.'}

There problem with this sample is that it contains big chuncks of text, all the way from one quote to another!
- We have: `Who is  Usshaa ?`
- and we have: `Usshaa  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.`

While for fine-tuning, we need these chunks to be much smaller! Not sentence long, but more like a word or half-a-world long! To accomplish that, we need something called "tokenization".

## Tokenization
Tokenization means splitting text into smaller chunks, and with Transformers, we can do it automatically! Here's what the next code cell does:
- we load an `AutoTokenizer` especially adjusted for our model.
- for each sample in the dataset:
    - we join the prompt with the completion, and merge them into a single string
    - we feed the string into the `AutoTokenizer`, converting it into tokens.
    - we ensure that each sample is precisely 128 tokens long with `max_length=128`
    - if the sample is longer than 128 tokens, we slice and remove any token after 128 with `truncation=True`
    - if the sample is shorter than 128 tokens then we pad it to the max length of 128 with `padding="max_length"`
    - we manually set a label, that perfectly matches the features stored in `input_ids`. <br>Yes, for text generation, our features and labels are the same!

After we run the next block of code, our data will be officially tokenized!

In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

def preprocess(sample):
    sample = sample["prompt"] + "\n" + sample["completion"]

    tokenized = tokenizer(
        sample,
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(preprocess)

Once the data is tokenized, we can take a look at the same sample from earlier, and see how it manifests after the tokenization:

### Preview Tokenized Sample

In [20]:
print(data["train"][0])

{'prompt': 'Who is  Usshaa ?', 'completion': 'Usshaa  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.', 'input_ids': [15191, 374, 220, 547, 778, 4223, 64, 17607, 52, 778, 4223, 64, 220, 374, 264, 23335, 323, 7988, 33968, 315, 12592, 85087, 11, 3881, 369, 1059, 5538, 6540, 323, 11438, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151

We notice a few things:
- Tokens are not words, but numbers! or more like numbers that represent words! each word (or half a word) has a unique token.
- The token we used for padding is 151643. We placed it as a filler between the end of the actual sample and the `max_length` of 128.
- Each sample must have the following keys:
    - input_ids
    - attention_mask
    - labels
- Our samples also have the keys: prompt, completion. They were kept by the `.map()` method.
  
## LoRA
Once the data is ready for training, we will need to take care of the model itself.
<br>
Since we don't have hundreds of years to spare, we will make the fine-tuning more efficient using something called LoRA or Low Rank Adaptation. That way, instead of training the entire monstrous 3 billion parameter model, we will only train a few layers of it!
<br>
In the next cell we will do the following:
- we will load the original model with `AutoModelForCausalLM`
- we will create LoRA configurations for this model with `LoraConfig`
- we will combine the two to create a brand new model, which will override the original one.

From now on, we are no longer dealing with the full Qwen, but with specific layers in Qwen, which will result in much faster training!

In [21]:
import bitsandbytes
print(bitsandbytes.__version__)

0.49.2


In [22]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# Load the model with the config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    quantization_config=bnb_config
)

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    target_modules = ["q_proj", "k_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Training / Fine Tuning

Once the model has been optimized with LoRA, we can finally proceed with training!
Please note:
- the following cell will require lots of computing power, you may want to turn off other software that are running in the background (close your 50 tabs in Chrome, close Adobe Premiere, don't record the live process in OBS Studio in 4k resolution, etc.).
- it takes about 10 minutes on GPUs with 16GB of VRAM.
- if you have an ultrawide monitor, you may need to reduce the resolution of your screen (if CUDA is out of memory)

Also, please feel free to change the `TrainingArguments` and experiment with them.

In [23]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [24]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./trainer_output",
    num_train_epochs=3,
    learning_rate=0.001,
    logging_steps=25,
    gradient_checkpointing=True,  # <--- ADD THIS
    per_device_train_batch_size=1, # <--- ENSURE THIS IS 1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"]
)

trainer.train()

Step,Training Loss
25,3.327222
50,0.616259
75,0.428027
100,0.355605
125,0.333876
150,0.306597
175,0.312816
200,0.288689
225,0.250681
250,0.256376


TrainOutput(global_step=708, training_loss=0.30641260810491056, metrics={'train_runtime': 481.5324, 'train_samples_per_second': 1.47, 'train_steps_per_second': 1.47, 'total_flos': 1510129614716928.0, 'train_loss': 0.30641260810491056, 'epoch': 3.0})

## Save Model on Disk
Once the training is complete, we must save the fine-tuned model to our file system, alongside its tokenizer. A new folder named `my_qwen` will be created at the root directory.

In [25]:
trainer.save_model("my_qwen")
tokenizer.save_pretrained("my_qwen")

('my_qwen/tokenizer_config.json',
 'my_qwen/chat_template.jinja',
 'my_qwen/tokenizer.json')

## Test Fine-Tuned Model
Finally, we will test if our training worked, asking our custom version of Qwen if it knows who I am.
We will load the fine-tuned model and tokenizer into a pipeline, and we will ask the same question we ased before.
<br>
<br>
**PLEASE NOTE:** the following code is a fix of the incorrect inference performed at the end of the video tutorial.
<br>
The previous implemintation only included the weights of the newly-learned data, ignoring the existing knowladge that the original model had!

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

path = "my_qwen"

config = PeftConfig.from_pretrained(path)
base = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base, path)

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)

inputs = tokenizer("Who is Usshaa?", return_tensors="pt").to(model.device)

output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
)

print(tokenizer.decode(output[0]))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=27) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Who is Usshaa?  Usshaa  is a wise and powerful wizard of Gondor, known for her


### congratulations!

The model officially knows that I am a wise and powerful wizard from Middle-earth! 😉
Fine tuning worked!!!

In [27]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", 
    device_map="cpu", # Use CPU to merge to avoid OOM
    trust_remote_code=True
)

# 2. Load and merge the LoRA adapters
model = PeftModel.from_pretrained(base_model, "my_qwen")
model = model.merge_and_unload()

# 3. Save the merged model
model.save_pretrained("my_qwen_merged")
tokenizer = AutoTokenizer.from_pretrained("my_qwen")
tokenizer.save_pretrained("my_qwen_merged")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_qwen_merged/tokenizer_config.json',
 'my_qwen_merged/chat_template.jinja',
 'my_qwen_merged/tokenizer.json')

In [31]:
from huggingface_hub import login
login() # Follow the prompt to paste your token

In [33]:
from huggingface_hub import HfApi

api = HfApi()

# 1. Create the repository
# Set private=False if you want it to be public
api.create_repo(repo_id="usshaa/my-qwen-finetuned3", repo_type="model", private=False)

# 2. Now run the upload
api.upload_folder(
    folder_path="my_qwen_merged",
    repo_id="usshaa/my-qwen-finetuned",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/usshaa/my-qwen-finetuned/commit/6d9254554f00c9cba44fc56e2e0e607d3e3115d6', commit_message='Upload folder using huggingface_hub', commit_description='', oid='6d9254554f00c9cba44fc56e2e0e607d3e3115d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/usshaa/my-qwen-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='usshaa/my-qwen-finetuned'), pr_revision=None, pr_num=None)

In [34]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load your model from the Hub
model_id = "usshaa/my-qwen-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# Test it
inputs = tokenizer("Who is Usshaa?", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Who is Usshaa?  Usshaa  is a wise and powerful wizard of Gondor, known for her deep knowledge and leadership.
